In [17]:
import os
import numpy as np
import onnx
import onnxruntime as ort

from transformers import AutoTokenizer

import aimet_onnx
from aimet_onnx.common.defs import QuantScheme
from aimet_onnx import QuantizationSimModel

#FP32_ONNX = "/root/AIMET_ruri/ruri-onnx/model_simplified.onnx"
FP32_ONNX = "/root/AIMET_ruri/ruri-onnx/model_simplified.onnx"
#FP32_ONNX = "/root/AIMET_ruri/ruri-onnx/model.onnx"  # ← 適宜変更
OUT_DIR = "/root/AIMET_ENV/tools/ruri/model"

os.makedirs(OUT_DIR, exist_ok=True)

MODEL_ID = "/root/AIMET_ruri/ruri-small-v2"
# tokenizer は事前にロード
tokenizer = AutoTokenizer.from_pretrained(
    "/root/AIMET_ruri/ruri-small-v2",
    trust_remote_code=True,
)

# 1) load onnx
model = onnx.load_model(FP32_ONNX)

# 2) (推奨) simplify
try:
    import onnxsim
    model, _ = onnxsim.simplify(model)
except Exception as e:
    print("onnxsim simplify failed, continue with original model:", repr(e))

    
# 3) create QuantSim (CPU only, W8/A16)
providers = ["CPUExecutionProvider"]
sim = QuantizationSimModel(

    model,
    param_type=aimet_onnx.int8,
    activation_type=aimet_onnx.int16,
    #quant_scheme=QuantScheme.min_max,
    quant_scheme=QuantScheme.post_training_tf_enhanced,
    #config_file="default",
    config_file="htp_v73",
    #config_file="/root/AIMET_ruri_v2/tool/quantsim_fp_escape.json",  # ← 適宜変更
    providers=providers,
)



2026-04-22 04:55:16,560 - Quant - INFO - Selecting DefaultOpInstanceConfigGenerator to compute the specialized config. hw_version:V73


In [18]:
# ONNX input names (must match tokenizer outputs)
onnx_input_names = [i.name for i in sim.model.model.graph.input]
print("ONNX inputs:", onnx_input_names)

# 4) calibration data (代表テキスト。実運用に近い文を500〜1000程度推奨)
"""
calib_texts = [
    "クエリ: 瑠璃色はどんな色？",
    "文章: 瑠璃色（るりいろ）は、紫みを帯びた濃い青。名は、半貴石の瑠璃（ラピスラズリ、英: lapis lazuli）による。JIS慣用色名では「こい紫みの青」（略号 dp-pB）と定義している[1][2]。",
    "クエリ: ワシやタカのように、鋭いくちばしと爪を持った大型の鳥類を総称して「何類」というでしょう?",
    "文章: ワシ、タカ、ハゲワシ、ハヤブサ、コンドル、フクロウが代表的である。これらの猛禽類はリンネ前後の時代(17~18世紀)には鷲類・鷹類・隼類及び梟類に分類された。ちなみにリンネは狩りをする鳥を単一の目(もく)にまとめ、vultur(コンドル、ハゲワシ)、falco(ワシ、タカ、ハヤブサなど)、strix(フクロウ)、lanius(モズ)の4属を含めている。",
] * 200  # 例: 900文
"""
"""
queries = [
    "クエリ: 瑠璃色はどんな色？",
     "クエリ: ワシやタカのように、鋭いくちばしと爪を持った大型の鳥類を総称して「何類」というでしょう?",
] * 300

passages = [
     "文章: 瑠璃色（るりいろ）は、紫みを帯びた濃い青。名は、半貴石の瑠璃（ラピスラズリ、英: lapis lazuli）による。JIS慣用色名では「こい紫みの青」（略号 dp-pB）と定義している[1][2]。",
     "文章: ワシ、タカ、ハゲワシ、ハヤブサ、コンドル、フクロウが代表的である。これらの猛禽類はリンネ前後の時代(17~18世紀)には鷲類・鷹類・隼類及び梟類に分類された。ちなみにリンネは狩りをする鳥を単一の目(もく)にまとめ、vultur(コンドル、ハゲワシ)、falco(ワシ、タカ、ハヤブサなど)、strix(フクロウ)、lanius(モズ)の4属を含めている。",
] * 120

calib_texts = queries + passages



"""
from datasets import Dataset 
arrow_path = "/root/test_qut/data/vocab800/article_200_exe.arrow"  # ← 適宜変更
ds = Dataset.from_file(arrow_path)

calib_texts = ds["text"]  # ← "text" 列を適宜変更



def make_feed(text: str, max_length: int = 512):
    enc = tokenizer(
        text,
        padding="max_length",
        truncation=True,
        max_length=512,
        return_tensors="np",
    )

    feed = {}

    # input_ids / attention_mask / token_type_ids
    if "input_ids" in onnx_input_names:
        feed["input_ids"] = enc["input_ids"].astype(np.int64)

    if "attention_mask" in onnx_input_names:
        feed["attention_mask"] = enc["attention_mask"].astype(np.int64)

    if "token_type_ids" in onnx_input_names:
        if "token_type_ids" in enc:
            feed["token_type_ids"] = enc["token_type_ids"].astype(np.int64)
        else:
            # tokenizer が返さない場合は 0 埋め
            feed["token_type_ids"] = np.zeros_like(
                enc["input_ids"], dtype=np.int64
            )

    # position_ids を明示的に追加
    if "position_ids" in onnx_input_names:
        batch_size, seq_len = enc["input_ids"].shape
        feed["position_ids"] = np.tile(
            np.arange(seq_len, dtype=np.int64),
            (batch_size, 1),
        )

    # 念のためチェック
    missing = [n for n in onnx_input_names if n not in feed]
    if missing:
        raise RuntimeError(
            f"Missing ONNX inputs {missing}. tokenizer keys={list(enc.keys())}"
        )

    return feed

def calib_generator(texts, num_batches: int = 500):
    # aimet_onnx の compute_encodings は iterable を受け取れる（dictをyield）
    for i, t in enumerate(texts):
        if i >= num_batches:
            break
        yield make_feed(t)






# 5) compute encodings
# 代表データ 500～1000サンプルが目安（AIMET docsでもそのレンジ）
sim.compute_encodings(calib_generator(calib_texts, num_batches=600))

ONNX inputs: ['input_ids', 'attention_mask', 'token_type_ids', 'position_ids']


In [22]:
# 6) export (QDQ onnx + encodings json)
OUT_DIR = "/root/AIMET_ENV/tools/ruri/model"

sim.export(
    path=OUT_DIR,
    filename_prefix="ruri_ptq_quat_article",
    export_model=True,
    export_int32_bias=True,    # 迷ったらTrueでOK（INT32 bias encodingを生成）
    encoding_version="1.0.0",  # デフォルトでも可
)

print("Done. Exported to:", OUT_DIR)
print("QDQ model:", os.path.join(OUT_DIR, "ruri_ptq_quat_article.onnx"))
print("Encodings:", os.path.join(OUT_DIR, "ruri_ptq_quat_article.encodings"))

Done. Exported to: /root/AIMET_ENV/tools/ruri/model
QDQ model: /root/AIMET_ENV/tools/ruri/model/ruri_ptq_quat_article.onnx
Encodings: /root/AIMET_ENV/tools/ruri/model/ruri_ptq_quat_article.encodings


In [23]:
import sys
from pathlib import Path
import numpy as np

# JMTEB v1 を pip install せず import
JMTEB_DIR = Path("/root/JMTEB")
sys.path.insert(0, str(JMTEB_DIR / "src"))

from jmteb.evaluators.retrieval.data import HfRetrievalQueryDataset, HfRetrievalDocDataset
from jmteb.evaluators.retrieval.evaluator import RetrievalEvaluator


class AimetOnnxEmbedder:
    def __init__(self, sim, tokenizer, session=None, max_length=512, pool="mean"):
        self.sim = sim
        self.tokenizer = tokenizer
        self.session = session
        self.max_length = max_length
        self.pool = pool

    def _pool(self, last_hidden_state, attention_mask):
        if self.pool == "cls":
            return last_hidden_state[:, 0, :]
        mask = attention_mask.astype(np.float32)
        denom = np.clip(mask.sum(axis=1, keepdims=True), 1e-6, None)
        return (last_hidden_state * mask[:, :, None]).sum(axis=1) / denom

    def encode(self, sentences, batch_size=1, **kwargs):
        embs = []
        sess = self.session or self.sim.session
        required = {i.name for i in sess.get_inputs()}

        for i in range(0, len(sentences), batch_size):
            batch = sentences[i : i + batch_size]
            tok = self.tokenizer(
                batch,
                padding="max_length",   # ★固定512に必須
                truncation=True,
                max_length=512,         # ★Expected: 512 に合わせる
                return_tensors="np",
            )

            input_ids = tok["input_ids"].astype(np.int64)
            attention_mask = tok["attention_mask"].astype(np.int64)

            inputs = {
                "input_ids": input_ids,
                "attention_mask": attention_mask,
            }

            # token_type_ids（無いなら0埋め）
            if "token_type_ids" in required:
                if "token_type_ids" in tok:
                    inputs["token_type_ids"] = tok["token_type_ids"].astype(np.int64)
                else:
                    inputs["token_type_ids"] = np.zeros_like(input_ids, dtype=np.int64)

            # position_ids（無いなら作る）
            if "position_ids" in required:
                bsz, seqlen = input_ids.shape  # seqlen は 512 のはず
                pos = np.arange(seqlen, dtype=np.int64)[None, :]
                inputs["position_ids"] = np.repeat(pos, bsz, axis=0)

            outputs = sess.run(None, inputs)

            last_hidden_state = outputs[0]
            pooled = self._pool(last_hidden_state, attention_mask).astype(np.float32)
            pooled /= np.clip(np.linalg.norm(pooled, axis=1, keepdims=True), 1e-12, None)
            embs.append(pooled)

        return np.vstack(embs)


class TextEmbedderCompat:
    def __init__(self, embedder: AimetOnnxEmbedder, batch_size=1):
        self.embedder = embedder
        self.batch_size = batch_size

    def set_output_tensor(self):  # called by RetrievalEvaluator
        return

    def reset_max_seq_length(self):  # only used if force_max_length=True
        return

    def batch_encode_with_cache(self, text_list, prefix=None, cache_path=None, overwrite_cache=False, **kwargs):
        if prefix:
            text_list = [prefix + t for t in text_list]
        return self.embedder.encode(text_list, batch_size=self.batch_size)

import json
import dataclasses
from pathlib import Path

def _to_jsonable(obj):
    # dataclass -> dict
    if dataclasses.is_dataclass(obj):
        return {k: _to_jsonable(v) for k, v in dataclasses.asdict(obj).items()}
    # numpy -> python
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    if isinstance(obj, (np.floating, np.float32, np.float64)):
        return float(obj)
    if isinstance(obj, (np.integer, np.int32, np.int64)):
        return int(obj)
    # dict/list
    if isinstance(obj, dict):
        return {str(k): _to_jsonable(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [_to_jsonable(x) for x in obj]
    # その他はそのまま（JSONが無理ならstrにする）
    try:
        json.dumps(obj)
        return obj
    except TypeError:
        return str(obj)

def save_results(results, out_path: str):
    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    payload = _to_jsonable(results)
    out_path.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")
    print("saved:", out_path)

def main():
    DATASET_ROOT = "/root/jmteb_260326"  # ← save_to_disk() した親ディレクトリ

    query_ds = HfRetrievalQueryDataset(
        path="sbintuitions/JMTEB",  # ここはログ用に残ってるだけ
        name="nlp_journal_abs_intro-query",
        split="test",
        query_key="query",
        relevant_docs_key="relevant_docs",
        dataset_path=DATASET_ROOT,  # ★必須
    )

    doc_ds = HfRetrievalDocDataset(
        path="sbintuitions/JMTEB",
        name="nlp_journal_abs_intro-corpus",
        split="corpus",
        id_key="docid",
        text_key="text",
        dataset_path=DATASET_ROOT,  # ★必須
    )

    evaluator = RetrievalEvaluator(
        val_query_dataset=query_ds,
        test_query_dataset=query_ds,  # valが無いので同じもの
        doc_dataset=doc_ds,
        doc_chunk_size=200000,
        log_predictions=False,
    )

    # sim/tokenizer をここで用意
    # sim = QuantizationSimModel(...)
    # tokenizer = AutoTokenizer.from_pretrained(...)
    model = TextEmbedderCompat(AimetOnnxEmbedder(sim=sim, tokenizer=tokenizer), batch_size=1)

    results = evaluator(model, cache_dir="cache_abs_intro", overwrite_cache=False)
    print(results)
    save_results(results, "results/abs_intro_retrieval_aimet_816.json")
if __name__ == "__main__":
    main()

2026-04-22 08:20:37.281 | INFO     | jmteb.evaluators.retrieval.data:__init__:153 - Loading dataset sbintuitions/JMTEB (name=nlp_journal_abs_intro-corpus) with split corpus
2026-04-22 08:22:32.800 | INFO     | jmteb.evaluators.retrieval.evaluator:__call__:165 - Start retrieval
Retrieval doc chunks: 100%|██████████| 637/637 [00:00<00:00, 185875.31it/s]


EvaluationResults(metric_name='ndcg@10', metric_value=0.8940849441978039, details={'optimal_distance_metric': 'cosine_similarity', 'val_scores': {'cosine_similarity': {'accuracy@1': 0.8215686274509804, 'accuracy@3': 0.9117647058823529, 'accuracy@5': 0.9450980392156862, 'accuracy@10': 0.9607843137254902, 'accuracy@20': 0.9725490196078431, 'accuracy@30': 0.9784313725490196, 'accuracy@50': 0.9823529411764705, 'ndcg@10': np.float64(0.8940849441978039), 'mrr@50': 0.8733968558429567}, 'dot_score': {'accuracy@1': 0.8215686274509804, 'accuracy@3': 0.9117647058823529, 'accuracy@5': 0.9450980392156862, 'accuracy@10': 0.9607843137254902, 'accuracy@20': 0.9725490196078431, 'accuracy@30': 0.9784313725490196, 'accuracy@50': 0.9823529411764705, 'ndcg@10': np.float64(0.8940849441978039), 'mrr@50': 0.8733968558429567}, 'euclidean_distance': {'accuracy@1': 0.8215686274509804, 'accuracy@3': 0.9117647058823529, 'accuracy@5': 0.9450980392156862, 'accuracy@10': 0.9607843137254902, 'accuracy@20': 0.972549019

In [11]:
# JMTEB v1 を pip install せず import
JMTEB_DIR = Path("/root/JMTEB")
sys.path.insert(0, str(JMTEB_DIR / "src"))

from jmteb.evaluators.retrieval.data import HfRetrievalQueryDataset, HfRetrievalDocDataset
from jmteb.evaluators.retrieval.evaluator import RetrievalEvaluator


class OnnxEmbedder:
    def __init__(self, session, tokenizer, max_length=512, pool="mean"):
        self.session = session
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.pool = pool

    def _pool(self, last_hidden_state, attention_mask):
        if self.pool == "cls":
            return last_hidden_state[:, 0, :]
        mask = attention_mask.astype(np.float32)
        denom = np.clip(mask.sum(axis=1, keepdims=True), 1e-6, None)
        return (last_hidden_state * mask[:, :, None]).sum(axis=1) / denom

    def encode(self, sentences, batch_size=1, **kwargs):
        embs = []
        required = {i.name for i in self.session.get_inputs()}

        for i in range(0, len(sentences), batch_size):
            batch = sentences[i : i + batch_size]
            tok = self.tokenizer(
                batch,
                padding="max_length",
                truncation=True,
                max_length=self.max_length,
                return_tensors="np",
            )

            input_ids = tok["input_ids"].astype(np.int64)
            attention_mask = tok["attention_mask"].astype(np.int64)

            inputs = {
                "input_ids": input_ids,
                "attention_mask": attention_mask,
            }

            if "token_type_ids" in required:
                if "token_type_ids" in tok:
                    inputs["token_type_ids"] = tok["token_type_ids"].astype(np.int64)
                else:
                    inputs["token_type_ids"] = np.zeros_like(input_ids, dtype=np.int64)

            if "position_ids" in required:
                bsz, seqlen = input_ids.shape
                pos = np.arange(seqlen, dtype=np.int64)[None, :]
                inputs["position_ids"] = np.repeat(pos, bsz, axis=0)

            outputs = self.session.run(None, inputs)
            last_hidden_state = outputs[0]

            pooled = self._pool(last_hidden_state, attention_mask).astype(np.float32)
            pooled /= np.clip(np.linalg.norm(pooled, axis=1, keepdims=True), 1e-12, None)
            embs.append(pooled)

        return np.vstack(embs)


def main_onnx():
    DATASET_ROOT = "/root/jmteb_260326"

    query_ds = HfRetrievalQueryDataset(
        path="sbintuitions/JMTEB",
        name="nlp_journal_abs_intro-query",
        split="test",
        query_key="query",
        relevant_docs_key="relevant_docs",
        dataset_path=DATASET_ROOT,
    )

    doc_ds = HfRetrievalDocDataset(
        path="sbintuitions/JMTEB",
        name="nlp_journal_abs_intro-corpus",
        split="corpus",
        id_key="docid",
        text_key="text",
        dataset_path=DATASET_ROOT,
    )

    evaluator = RetrievalEvaluator(
        val_query_dataset=query_ds,
        test_query_dataset=query_ds,
        doc_dataset=doc_ds,
        doc_chunk_size=200000,
        log_predictions=False, 
    )
    
    onnx_model_path = os.path.join("/root/AIMET_ruri_v2/ruri-onnx_vvv/model.onnx")

    session = ort.InferenceSession(onnx_model_path, providers=providers)
    print("Loaded ONNX:", onnx_model_path)
    print("Inputs:", [x.name for x in session.get_inputs()])
    print("Outputs:", [x.name for x in session.get_outputs()])

    model = TextEmbedderCompat(
        OnnxEmbedder(session=session, tokenizer=tokenizer, max_length=512, pool="mean"),
        batch_size=1,
    )

    results = evaluator(model, cache_dir="cache_abs_intro_onnx", overwrite_cache=False)
    print(results)
    save_results(results, "results/abs_intro_retrieval_onnx.json")


main_onnx()

2026-04-22 04:15:15.337 | INFO     | jmteb.evaluators.retrieval.data:__init__:153 - Loading dataset sbintuitions/JMTEB (name=nlp_journal_abs_intro-corpus) with split corpus


Loaded ONNX: /root/AIMET_ruri_v2/ruri-onnx_vvv/model.onnx
Inputs: ['input_ids', 'attention_mask']
Outputs: ['last_hidden_state']


2026-04-22 04:15:58.282 | INFO     | jmteb.evaluators.retrieval.evaluator:__call__:165 - Start retrieval
Retrieval doc chunks: 100%|██████████| 637/637 [00:00<00:00, 15467.91it/s]


EvaluationResults(metric_name='ndcg@10', metric_value=0.9044389931055391, details={'optimal_distance_metric': 'cosine_similarity', 'val_scores': {'cosine_similarity': {'accuracy@1': 0.8470588235294118, 'accuracy@3': 0.9274509803921569, 'accuracy@5': 0.9372549019607843, 'accuracy@10': 0.9588235294117647, 'accuracy@20': 0.9705882352941176, 'accuracy@30': 0.9764705882352941, 'accuracy@50': 0.984313725490196, 'ndcg@10': np.float64(0.9044389931055391), 'mrr@50': 0.8881360771421324}, 'dot_score': {'accuracy@1': 0.8470588235294118, 'accuracy@3': 0.9274509803921569, 'accuracy@5': 0.9372549019607843, 'accuracy@10': 0.9588235294117647, 'accuracy@20': 0.9705882352941176, 'accuracy@30': 0.9764705882352941, 'accuracy@50': 0.984313725490196, 'ndcg@10': np.float64(0.9044389931055391), 'mrr@50': 0.8881360771421324}, 'euclidean_distance': {'accuracy@1': 0.8470588235294118, 'accuracy@3': 0.9274509803921569, 'accuracy@5': 0.9372549019607843, 'accuracy@10': 0.9588235294117647, 'accuracy@20': 0.97058823529

In [26]:
import numpy as np

def get_embedding(text: str, sim=sim, tokenizer=tokenizer, max_length=512, pool="mean"):
    """
    テキストをsimモデルに渡してembeddingベクトルを返す
    """
    required = {i.name for i in sim.session.get_inputs()}
    
    tok = tokenizer(
        [text],
        padding="max_length",
        truncation=True,
        max_length=max_length,
        return_tensors="np",
    )
    
    input_ids = tok["input_ids"].astype(np.int64)
    attention_mask = tok["attention_mask"].astype(np.int64)
    
    inputs = {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
    }
    
    if "token_type_ids" in required:
        if "token_type_ids" in tok:
            inputs["token_type_ids"] = tok["token_type_ids"].astype(np.int64)
        else:
            inputs["token_type_ids"] = np.zeros_like(input_ids, dtype=np.int64)
    
    if "position_ids" in required:
        bsz, seqlen = input_ids.shape
        inputs["position_ids"] = np.tile(np.arange(seqlen, dtype=np.int64), (bsz, 1))
    
    # 推論実行
    outputs = sim.session.run(None, inputs)
    last_hidden_state = outputs[0]  # shape: (1, seq_len, hidden_size)
    
    # pooling
    if pool == "cls":
        pooled = last_hidden_state[:, 0, :]
    else:  # mean pooling
        mask = attention_mask.astype(np.float32)
        denom = np.clip(mask.sum(axis=1, keepdims=True), 1e-6, None)
        pooled = (last_hidden_state * mask[:, :, None]).sum(axis=1) / denom
    
    # 正規化
    pooled = pooled.astype(np.float32)
    pooled /= np.clip(np.linalg.norm(pooled, axis=1, keepdims=True), 1e-12, None)
    
    # output.raw 相当: shape (hidden_size,) の numpy array
    output_raw = pooled[0]
    return output_raw


# 使用例
text = "クエリ: 瑠璃色はどんな色？"
output_raw = get_embedding(text)
print("output.raw shape:", output_raw.shape)
print("output.raw:", output_raw[:10], "...")  # 先頭10次元を表示
output_raw.astype(np.float32).tofile("/root/AIMET_ENV/tools/ruri/convert/sim_output.raw")

output.raw shape: (768,)
output.raw: [-0.03384766  0.03208147 -0.04087752 -0.00207943  0.00784999 -0.01456555
  0.00382023 -0.00135877 -0.01606507  0.01079294] ...


In [ ]:
# 2つのRAWファイルを読み込んで比較
raw_path1 = "/root/AIMET_ENV/tools/ruri/convert/sim_output.raw"
raw_path2 = "/root/AIMET_ENV/tools/ruri/convert/sim_output2.raw"

vec1 = np.fromfile(raw_path1, dtype=np.float32)
vec2 = np.fromfile(raw_path2, dtype=np.float32)

# コサイン類似度計算
def cosine_similarity(a, b):
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-12))

sim_score = cosine_similarity(vec1, vec2)
print(f"類似度: {sim_score:.4f}")


vec1 shape: (768,)

基準テキスト: 'クエリ: 瑠璃色はどんな色？'
------------------------------------------------------------
類似度: 1.0000  |  'クエリ: 瑠璃色はどんな色？'
類似度: 0.9497  |  '文章: 瑠璃色（るりいろ）は、紫みを帯びた濃い青。名は、半貴石の瑠璃（ラピスラズ...' 
類似度: 0.8172  |  'クエリ: ワシやタカのように、鋭いくちばしと爪を持った大型の鳥類を総称して「何類...' 
類似度: 0.8128  |  '文章: 全く関係のない文章です。今日の天気は晴れです。'
